# Tutorial 3: Scaling Strategies with Providers

## What You Will Learn

- Scalable's provider architecture and how it abstracts backends
- Configure the Local provider for development
- Session-based scaling with objectives and policies
- Heterogeneous worker pools with multiple components
- Adaptive scaling concepts

## Prerequisites

- Completed Tutorials 1 and 2
- `pip install scalable`

In [ ]:
import os
import tempfile
import time

project_dir = tempfile.mkdtemp(prefix="scalable-scaling-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Step 1: The Provider Architecture

Every provider implements the `DeploymentProvider` protocol:

```
Manifest → DeploymentSpec → Provider → Cluster
                                          ↓
                            Local / Slurm / Cloud / K8s
```

Your workflow code is **provider-agnostic** — the same `client.submit(func, arg, tag="gcam")` works identically regardless of backend.

In [ ]:
# List registered providers
from scalable.providers.registry import iter_provider_names

print("Registered providers:")
for name in iter_provider_names(include_entrypoints=True):
    print(f"  - {name}")

## Step 2: Local Provider — Development & CI

The `LocalProvider` wraps Dask's `LocalCluster`. It's the fastest way to iterate.

In [ ]:
manifest_content = """\
version: 1
project:
  name: scaling-demo

targets:
  local-threads:
    provider: local
    max_workers: 4
    threads_per_worker: 2
    processes: false
    containers: none

  local-processes:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: true
    containers: none

components:
  compute:
    cpus: 2
    memory: 4G
    tags: [heavy]

  io:
    cpus: 1
    memory: 1G
    tags: [light]

tasks:
  simulate:
    component: compute
    cache: true

  aggregate:
    component: io
    cache: false
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)

print("Manifest with two local targets (threads vs processes) written.")

### Threads vs Processes

| Mode | Startup | Isolation | Best For |
|------|---------|-----------|----------|
| `processes: false` | Fast | None (shared memory) | I/O-bound tasks, quick iteration |
| `processes: true` | Slower | Full (separate process) | CPU-bound tasks, GIL-heavy code |

In [ ]:
from scalable import ScalableSession

# Threaded mode (fast startup)
session = ScalableSession.from_yaml("./scalable.yaml", target="local-threads")
plan = session.plan(dry_run=True)

print(f"Target: {plan.target_name}")
print(f"Scale plan: {plan.scale_plan}")

## Step 3: Heterogeneous Worker Pools

Real workflows need different resource profiles running simultaneously. Submit to each pool independently using tags.

In [ ]:
def heavy_computation(scenario_id: int) -> dict:
    """CPU-intensive simulation."""
    time.sleep(0.5)
    return {"scenario": scenario_id, "result": scenario_id ** 2}


def light_aggregation(results: list) -> dict:
    """I/O-bound aggregation."""
    total = sum(r["result"] for r in results)
    return {"total": total, "count": len(results)}


# Start the cluster
client = session.start(plan)
print(f"Cluster started with {plan.scale_plan}")

In [ ]:
# Phase 1: Heavy compute tasks go to 'compute' workers
compute_futures = [
    client.submit(heavy_computation, i, tag="compute")
    for i in range(8)
]
print(f"Submitted {len(compute_futures)} compute tasks")

# Wait for compute results
compute_results = client.gather(compute_futures)
print(f"Compute results: {compute_results[:3]}...")

In [ ]:
# Phase 2: Light aggregation on 'io' workers
agg_future = client.submit(light_aggregation, compute_results, tag="io")
agg_result = agg_future.result()

print(f"Aggregation result: {agg_result}")

### Why Heterogeneous Pools?

This pattern avoids over-provisioning:
- Expensive 4 GB workers handle heavy computation
- Cheap 1 GB workers handle aggregation
- You only pay for the resources each task actually needs

## Step 4: Policy-Driven Planning

The Session API supports objectives that automatically influence worker allocation.

In [ ]:
session.close()

# Create a fresh session to demonstrate planning
session2 = ScalableSession.from_yaml("./scalable.yaml", target="local-threads")

# Minimize cost: fewest workers
plan_cost = session2.plan(objective="minimize cost", policy="safe")
print(f"Cost-optimized: {plan_cost.scale_plan}")

# Minimize time: max parallelism
plan_time = session2.plan(objective="minimize time", policy="aggressive")
print(f"Time-optimized: {plan_time.scale_plan}")

# Balance: midpoint
plan_balanced = session2.plan(objective="balance", policy="safe")
print(f"Balanced: {plan_balanced.scale_plan}")

## Step 5: AdaptiveScaler Concepts

For long-running workflows where task load varies, the `AdaptiveScaler` monitors queue depth and adjusts workers dynamically.

In [ ]:
try:
    from scalable import AdaptiveScaler
    
    scaler = AdaptiveScaler(
        min_workers={"compute": 1, "io": 1},
        max_workers={"compute": 10, "io": 5},
        scale_up_threshold=0.8,
        scale_down_threshold=0.2,
        cooldown_seconds=30,
    )
    
    # Simulate evaluation with pending tasks
    decision = scaler.evaluate(
        pending_tasks=[{"tag": "compute"} for _ in range(20)],
        active_workers={"compute": 2, "io": 1},
    )
    
    print(f"Scale decision:")
    print(f"  Has changes: {decision.has_changes}")
    print(f"  Workers to add: {decision.workers_to_add}")
    print(f"  Workers to remove: {decision.workers_to_remove}")
    print(f"  Reasoning: {decision.reasoning}")
    print(f"  Confidence: {decision.confidence:.2f}")
    
except ImportError:
    print("AdaptiveScaler requires scalable[ml]. Install with:")
    print("  pip install scalable[ml]")

## Step 6: Scaling Trade-offs

```
Aggressive (max workers)
├── Fastest completion
├── Highest cost
└── Risk: idle workers during low-load phases

Conservative (min workers)
├── Lowest cost
├── Slowest completion
└── Risk: queue buildup during bursts

Adaptive (dynamic scaling)
├── Best cost-performance ratio
├── Requires cooldown tuning
└── Latency: scale-up takes time
```

In [ ]:
# Demonstrate the impact of worker count on throughput
from scalable import ScalableSession
import time

def timed_workload(n_tasks, target_name):
    """Run n_tasks and measure total time."""
    sess = ScalableSession.from_yaml("./scalable.yaml", target=target_name)
    client = sess.start()
    
    start = time.time()
    futures = [client.submit(heavy_computation, i, tag="compute") for i in range(n_tasks)]
    results = client.gather(futures)
    elapsed = time.time() - start
    
    sess.close()
    return elapsed

# 4-worker threaded (from local-threads)
time_4w = timed_workload(8, "local-threads")
print(f"4 workers, 8 tasks: {time_4w:.2f}s")

# 2-worker process (from local-processes)
time_2w = timed_workload(8, "local-processes")
print(f"2 workers, 8 tasks: {time_2w:.2f}s")

print(f"\nSpeedup: {time_2w/time_4w:.1f}x")

## Summary

In this tutorial you learned:
1. The provider architecture abstracts execution backends
2. LocalProvider supports both threaded and process-based workers
3. Heterogeneous pools match resources to task requirements
4. Policy-driven planning automates worker count decisions
5. AdaptiveScaler provides real-time scaling recommendations

## Next Steps

- **Tutorial 4**: Cache expensive computations
- **Tutorial 5**: Deploy to AWS/GCP cloud providers
- **Tutorial 8**: Kubernetes provider with pod management

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")